In [ ]:
import pandas as pd

df = pd.read_csv('all-data.csv')

In [1]:
from datasets import Dataset

data = [
    {"text": "J'adore ce produit, il est génial !", "label": 0},
    {"text": "Ce film était horrible et ennuyeux", "label": 1},
    {"text": "Le service était correct, sans plus", "label": 2},
    {"text": "J'aime les pommes, c'est bon", "label": 0},
]

dataset = Dataset.from_list(data)

dataset = dataset.train_test_split(test_size=0.5)

C:\Users\laaro\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import LlamaForSequenceClassification, LlamaTokenizer, Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, LlamaConfig
import json
import os

model_path = r"C:/Users/laaro/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/221e3535e1ac4840bdf061a12b634139c84e144c"
#token_path = r"C:/Users/laaro/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/221e3535e1ac4840bdf061a12b634139c84e144c/tokenizer.model"

# Charger le fichier de configuration brut comme dictionnaire JSON
config_path = os.path.join(model_path, "config.json")
with open(config_path, "r") as f:
    config_dict = json.load(f)

# Configuration rope scaling
config_dict["rope_scaling"] = {"type": "linear", "factor": 32.0}  # Ajuster le `type` et `factor` si besoin
config = LlamaConfig.from_dict(config_dict)
config.num_labels = 3

# Modèle
model = LlamaForSequenceClassification.from_pretrained(model_path, config=config)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)
tokenized_data = dataset.map(tokenize_function, batched=True)

'''
# Charger et modifier la configuration
config = LlamaConfig.from_pretrained(model_path)
config.rope_scaling = {"type": "linear", "factor": 32.0}  # Ajuster `type` et `factor` selon le besoin

# Initialiser le modèle avec la configuration modifiée
model = LlamaForSequenceClassification.from_pretrained(model_path, config=config, num_labels=3)

#model = LlamaForSequenceClassification.from_pretrained(model_path, num_labels=3)
tokenizer = LlamaTokenizer.from_pretrained(model_path)
'''

# Paramètres
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['test'],
)

trainer.train()


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at C:/Users/laaro/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/221e3535e1ac4840bdf061a12b634139c84e144c and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map: 100%|██████████| 2/2 [00:00<00:00, 11.45 examples/s]

Map: 100%|██████████| 2/2 [00:00<00:00, 21.98 examples/s]



RuntimeError: [enforce fail at alloc_cpu.cpp:80] data. DefaultCPUAllocator: not enough memory: you tried to allocate 68719476736 bytes.

In [ ]:
results = trainer.evaluate()

print(results)


In [ ]:
text = "Ce produit est vraiment mauvais"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
predicted_class = logits.argmax().item()
